# معلومات
- تم تصفية الموديل من القيم المتكررة
- تم اضافة اعمدة او خواص جديدة للبيانات
- تم التعديل على خوارزمية وعدم اعتماد القيم التقليدية


# Import the libraries
- تم اضافة كل المكتبات التي نحتاجها

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix, classification_report, roc_auc_score
from google.colab import files
from flask import Flask, request, jsonify


# pandas

## mounting the data
- تحميل البيانات من الدرايف
- انشاء نسخه من البيانات والعمل عليها للحغاظ على البيانات الاصلية

In [2]:
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/dataset/data.csv"

df_original = pd.read_csv(path)
df = df_original.copy()
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,android.permission.GET_ACCOUNTS,com.sonyericsson.home.permission.BROADCAST_BADGE,android.permission.READ_PROFILE,android.permission.MANAGE_ACCOUNTS,android.permission.WRITE_SYNC_SETTINGS,android.permission.READ_EXTERNAL_STORAGE,android.permission.RECEIVE_SMS,com.android.launcher.permission.READ_SETTINGS,android.permission.WRITE_SETTINGS,com.google.android.providers.gsf.permission.READ_GSERVICES,...,com.android.launcher.permission.UNINSTALL_SHORTCUT,com.sec.android.iap.permission.BILLING,com.htc.launcher.permission.UPDATE_SHORTCUT,com.sec.android.provider.badge.permission.WRITE,android.permission.ACCESS_NETWORK_STATE,com.google.android.finsky.permission.BIND_GET_INSTALL_REFERRER_SERVICE,com.huawei.android.launcher.permission.READ_SETTINGS,android.permission.READ_SMS,android.permission.PROCESS_INCOMING_CALLS,Result
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,1,0,0,0,0


## data info
- عدد الصفوف والاعمدة
- معلومات كل عمود وهل يحتوي على قيم فالرغة ام لا ونوع البيانات
- مصفوفة تحتوي كل اسماء الاعمدة
-  القيم المتوسطة والاصغر والاكبر لكل عمود
- حساب القيم لعمود الهدف

In [3]:
# print("number of row and colums", df.shape)
#print("\n", "#" * 50 , "\n")
# df.info()
#print("\n", "#" * 50 , "\n")
# print(df.columns)
#print("\n", "#" * 50 , "\n")
# print(df.describe())
#print("\n", "#" * 50 , "\n")
# df['Result'].value_counts()

## cleaning the data
- تنضيف البيانات من القيم الخالية
- ازالة الصفوف المتكررة


In [4]:
print("the number of null values in the dataset",df_original.isnull().sum().sum())
df.dropna(inplace=True)
print("the number of duplicates in the dataset", df_original.duplicated().sum())
df.drop_duplicates(inplace=True)

df.isnull().sum().sort_values(ascending=False).head(10)


the number of null values in the dataset 0
the number of duplicates in the dataset 21841


,0
android.permission.GET_ACCOUNTS,0
com.sonyericsson.home.permission.BROADCAST_BADGE,0
android.permission.READ_PROFILE,0
android.permission.MANAGE_ACCOUNTS,0
android.permission.WRITE_SYNC_SETTINGS,0
android.permission.READ_EXTERNAL_STORAGE,0
android.permission.RECEIVE_SMS,0
com.android.launcher.permission.READ_SETTINGS,0
android.permission.WRITE_SETTINGS,0
com.google.android.providers.gsf.permission.READ_GSERVICES,0


In [5]:
print("the original data", df_original.shape)
print("the working data", df.shape)

the original data (29332, 87)
the working data (7491, 87)


In [6]:
print(df['Result'].value_counts())
print("#" * 50)
df.dtypes.value_counts()


Result
0    4867
1    2624
Name: count, dtype: int64
##################################################


,count
int64,87


# Scikit learn
- تقسيم البيانات لكي يتسنى لنا التعالمل معها كبيانات تدريب وبيانات هدف

In [7]:
X = df.drop(columns=['Result'])
y = df['Result']

print(X.shape)
print(y.shape)
print(y.head())


(7491, 86)
(7491,)
0    0
1    0
2    0
3    0
4    0
Name: Result, dtype: int64


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)


(5992, 86)
(1499, 86)
(5992,)
(1499,)


# The Model

## RF 1
* **100 trees** → Good accuracy/speed balance; performance plateaus after this.
* **max_depth=None** → Deep trees capture complex, non-linear malware patterns.
* **n_jobs=-1** → Uses all CPU cores; trees train in parallel.


In [9]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print(y_pred_rf[:10])
print(y_test.values[:10])

print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

[0 0 0 1 1 0 0 1 0 0]
[0 0 0 1 1 0 0 1 0 0]
[[970  45]
 [ 47 437]]
              precision    recall  f1-score   support

           0       0.95      0.96      0.95      1015
           1       0.91      0.90      0.90       484

    accuracy                           0.94      1499
   macro avg       0.93      0.93      0.93      1499
weighted avg       0.94      0.94      0.94      1499



### using threshold to decrease the FN

In [10]:
y_proba = rf.predict_proba(X_test)[:, 1]
threshold = 0.3
y_pred_custom = (y_proba >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))


[[906 109]
 [ 27 457]]
              precision    recall  f1-score   support

           0       0.97      0.89      0.93      1015
           1       0.81      0.94      0.87       484

    accuracy                           0.91      1499
   macro avg       0.89      0.92      0.90      1499
weighted avg       0.92      0.91      0.91      1499



In [11]:
print(df.columns.tolist())
print("\nNew features exist?")
print(all(col in df.columns for col in [
    'number_of_permissions',
    'sensitive_permission_count',
    'network_permission_ratio',
    'persistence_permission_count',
    'low_perm_high_net',
    'stealth_data_access',
    'minimal_malware_pattern']))

['android.permission.GET_ACCOUNTS', 'com.sonyericsson.home.permission.BROADCAST_BADGE', 'android.permission.READ_PROFILE', 'android.permission.MANAGE_ACCOUNTS', 'android.permission.WRITE_SYNC_SETTINGS', 'android.permission.READ_EXTERNAL_STORAGE', 'android.permission.RECEIVE_SMS', 'com.android.launcher.permission.READ_SETTINGS', 'android.permission.WRITE_SETTINGS', 'com.google.android.providers.gsf.permission.READ_GSERVICES', 'android.permission.DOWNLOAD_WITHOUT_NOTIFICATION', 'android.permission.GET_TASKS', 'android.permission.WRITE_EXTERNAL_STORAGE', 'android.permission.RECORD_AUDIO', 'com.huawei.android.launcher.permission.CHANGE_BADGE', 'com.oppo.launcher.permission.READ_SETTINGS', 'android.permission.CHANGE_NETWORK_STATE', 'com.android.launcher.permission.INSTALL_SHORTCUT', 'android.permission.android.permission.READ_PHONE_STATE', 'android.permission.CALL_PHONE', 'android.permission.WRITE_CONTACTS', 'android.permission.READ_PHONE_STATE', 'com.samsung.android.providers.context.permi

## Features selection

### FN = 27
- The FN still high at 27 even after using threshold, now need the common permisions between those 27 malware
- safe them as an index
- look at thire features

In [12]:
# save the FN in an index (save the 27 rows and thier features)
fn_idx = np.where((y_test == 1) & (y_pred_custom == 0))[0]
len(fn_idx)

27

In [13]:
# extracting the features and sorting them
importances = rf.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

feature_importance_df.head(10)

,Feature,Importance
21,android.permission.READ_PHONE_STATE,0.195079
39,com.google.android.c2dm.permission.RECEIVE,0.150187
17,com.android.launcher.permission.INSTALL_SHORTCUT,0.093702
65,android.permission.RECEIVE_BOOT_COMPLETED,0.041386
5,android.permission.READ_EXTERNAL_STORAGE,0.039799
57,android.permission.SYSTEM_ALERT_WINDOW,0.033574
63,com.android.vending.BILLING,0.031127
11,android.permission.GET_TASKS,0.029258
53,android.permission.ACCESS_COARSE_LOCATION,0.025039
66,android.permission.WAKE_LOCK,0.022317


In [14]:
# what are the most importantent features for the 27 FNs
fn_samples = X_test.iloc[fn_idx]
# print(fn_samples)
fn_samples.mean().sort_values(ascending=False).head(10)

,0
android.permission.INTERNET,1.000000
android.permission.ACCESS_NETWORK_STATE,0.925926
android.permission.WRITE_EXTERNAL_STORAGE,0.814815
android.permission.READ_PHONE_STATE,0.629630
android.permission.ACCESS_WIFI_STATE,0.592593
android.permission.WAKE_LOCK,0.370370
android.permission.VIBRATE,0.370370
android.permission.GET_ACCOUNTS,0.296296
com.google.android.c2dm.permission.RECEIVE,0.259259
com.android.vending.BILLING,0.222222


### adding the features

In [15]:
# permission_columns = df.drop('Result', axis=1).columns # it did not worked
permission_columns = [col for col in df.columns if 'permission' in col.lower()]

sensitive_perms = [
    'android.permission.READ_PHONE_STATE',
    'android.permission.READ_EXTERNAL_STORAGE',
    'android.permission.WRITE_EXTERNAL_STORAGE'
]

network_perms = [
    'android.permission.INTERNET',
    'android.permission.ACCESS_NETWORK_STATE',
    'android.permission.ACCESS_WIFI_STATE',
    'com.google.android.c2dm.permission.RECEIVE'
]

persistence_perms = [
    'android.permission.RECEIVE_BOOT_COMPLETED',
    'android.permission.WAKE_LOCK',
    'android.permission.SYSTEM_ALERT_WINDOW'
]

binary_permissions = (df[permission_columns] > 0).astype(int)

df['number_of_permissions'] = binary_permissions.sum(axis=1)

df['sensitive_permission_count'] = binary_permissions[sensitive_perms].sum(axis=1)

df['network_permission_ratio'] = (
    binary_permissions[network_perms].sum(axis=1)
    / df['number_of_permissions'].replace(0,1)
)

df['persistence_permission_count'] = binary_permissions[persistence_perms].sum(axis=1)


df['low_perm_high_net'] = (
    (df['number_of_permissions'] <= 8) &
    (df['network_permission_ratio'] > 0.2)
).astype(int)


df['stealth_data_access'] = (
    (df['sensitive_permission_count'] >= 1) &
    (df['persistence_permission_count'] == 0)
).astype(int)

df['minimal_malware_pattern'] = (
    (df['number_of_permissions'] < 10) &
    (df['sensitive_permission_count'] >= 1)
).astype(int)


df[['number_of_permissions',
    'sensitive_permission_count',
    'network_permission_ratio',
    'persistence_permission_count',
    'low_perm_high_net',
    'stealth_data_access',
    'minimal_malware_pattern']].head(10)

,number_of_permissions,sensitive_permission_count,network_permission_ratio,persistence_permission_count,low_perm_high_net,stealth_data_access,minimal_malware_pattern
0,0,0,0.000000,0,0,0,0
1,7,2,0.428571,1,1,0,1
2,6,1,0.500000,1,1,0,1
3,3,0,1.000000,0,1,0,0
4,5,0,0.400000,1,1,0,0
5,7,2,0.428571,0,1,1,1
6,6,2,0.500000,0,1,1,1
7,13,2,0.307692,2,0,0,0
8,2,0,1.000000,0,1,0,0
9,23,3,0.173913,1,0,0,0


In [16]:
df[['number_of_permissions','network_permission_ratio']].describe()

,number_of_permissions,network_permission_ratio
count,7491.000000,7491.000000
mean,11.880123,0.299626
std,6.468867,0.133284
min,0.000000,0.000000
25%,8.000000,0.200000
50%,10.000000,0.272727
75%,14.000000,0.375000
max,63.000000,1.000000


### checking if the features has been add or not

In [17]:
print(df.columns.tolist())
print("\nNew features exist?")
print(all(col in df.columns for col in [
    'number_of_permissions',
    'sensitive_permission_count',
    'network_permission_ratio',
    'persistence_permission_count',
    'low_perm_high_net',
    'stealth_data_access',
    'minimal_malware_pattern']))

['android.permission.GET_ACCOUNTS', 'com.sonyericsson.home.permission.BROADCAST_BADGE', 'android.permission.READ_PROFILE', 'android.permission.MANAGE_ACCOUNTS', 'android.permission.WRITE_SYNC_SETTINGS', 'android.permission.READ_EXTERNAL_STORAGE', 'android.permission.RECEIVE_SMS', 'com.android.launcher.permission.READ_SETTINGS', 'android.permission.WRITE_SETTINGS', 'com.google.android.providers.gsf.permission.READ_GSERVICES', 'android.permission.DOWNLOAD_WITHOUT_NOTIFICATION', 'android.permission.GET_TASKS', 'android.permission.WRITE_EXTERNAL_STORAGE', 'android.permission.RECORD_AUDIO', 'com.huawei.android.launcher.permission.CHANGE_BADGE', 'com.oppo.launcher.permission.READ_SETTINGS', 'android.permission.CHANGE_NETWORK_STATE', 'com.android.launcher.permission.INSTALL_SHORTCUT', 'android.permission.android.permission.READ_PHONE_STATE', 'android.permission.CALL_PHONE', 'android.permission.WRITE_CONTACTS', 'android.permission.READ_PHONE_STATE', 'com.samsung.android.providers.context.permi

## 2
- after adding the features, we tune the RF
- n_estimators from 100 --> 600
- min_samples_leaf = 3
- class_weight={0:1, 1:5}

In [18]:
# ★ أضف هذا السطر ★
X = df.drop(columns=['Result'])   # تحديث X ليشمل الـ 93 عموداً
y = df['Result']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
rf_tuned = RandomForestClassifier(
    n_estimators=600,
    max_depth=None,
    min_samples_leaf=3,
    max_features='sqrt',
    class_weight={0:1, 1:5},
    random_state=42,
    n_jobs=-1
)

rf_tuned.fit(X_train, y_train)

y_pred_tuned = rf_tuned.predict(X_test)


print(confusion_matrix(y_test, y_pred_tuned))
print(classification_report(y_test, y_pred_tuned))


[[870 104]
 [ 19 506]]
              precision    recall  f1-score   support

           0       0.98      0.89      0.93       974
           1       0.83      0.96      0.89       525

    accuracy                           0.92      1499
   macro avg       0.90      0.93      0.91      1499
weighted avg       0.93      0.92      0.92      1499



### using thresholde to decrease the FN

In [19]:
y_prob_new = rf_tuned.predict_proba(X_test)[:,1]

threshold = 0.3
y_pred_custom = (y_prob_new >= threshold).astype(int)

print(confusion_matrix(y_test, y_pred_custom))
print(classification_report(y_test, y_pred_custom))

[[760 214]
 [  7 518]]
              precision    recall  f1-score   support

           0       0.99      0.78      0.87       974
           1       0.71      0.99      0.82       525

    accuracy                           0.85      1499
   macro avg       0.85      0.88      0.85      1499
weighted avg       0.89      0.85      0.86      1499



# Feature Selection 2.0
- find and sort the most and leaset importent features


In [20]:
importances = rf_tuned.feature_importances_
feat_imp = pd.Series(importances, index=X_train.columns)
feat_imp = feat_imp.sort_values(ascending=False)

print(feat_imp.head(20))
print(feat_imp.tail(20))

android.permission.READ_PHONE_STATE                           0.218443
com.google.android.c2dm.permission.RECEIVE                    0.196387
com.android.launcher.permission.INSTALL_SHORTCUT              0.050508
network_permission_ratio                                      0.044006
com.android.vending.BILLING                                   0.043718
android.permission.READ_EXTERNAL_STORAGE                      0.040388
sensitive_permission_count                                    0.037551
number_of_permissions                                         0.028887
android.permission.RECEIVE_BOOT_COMPLETED                     0.019624
com.google.android.providers.gsf.permission.READ_GSERVICES    0.019544
android.permission.SYSTEM_ALERT_WINDOW                        0.018622
android.permission.CAMERA                                     0.016162
persistence_permission_count                                  0.014809
android.permission.ACCESS_COARSE_LOCATION                     0.013815
androi

In [21]:
# low_feats = feat_imp[feat_imp < 0.001].index

# X_train_red = X_train.drop(columns=low_feats)
# X_test_red  = X_test.drop(columns=low_feats)

# rf_tuned.fit(X_train_red, y_train)

# Saving the model

In [22]:
final_model = {
    "model": rf_tuned,
    "threshold": 0.3,
    "features": X_train.columns.tolist()
}

with open("malware_rf_final.pkl", "wb") as f:
    pickle.dump(final_model, f)

print("Final model saved successfully ✅")


Final model saved successfully ✅


In [23]:
import os
os.listdir()

data = pickle.load(open("malware_rf_final.pkl","rb"))
feature_names = data["features"]

In [24]:
clean_features = [
    f.replace("android.permission.android.permission.", "android.permission.")
    for f in feature_names
]

In [25]:
import json

with open("features.json", "w") as f:
    json.dump(feature_names, f, indent=2)

files.download("features.json")

if isinstance(feature_names, dict):
    feature_names = feature_names["features"]

print(feature_names)

with open("malware_rf_final.pkl", "rb") as f:
    model_test = pickle.load(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

['android.permission.GET_ACCOUNTS', 'com.sonyericsson.home.permission.BROADCAST_BADGE', 'android.permission.READ_PROFILE', 'android.permission.MANAGE_ACCOUNTS', 'android.permission.WRITE_SYNC_SETTINGS', 'android.permission.READ_EXTERNAL_STORAGE', 'android.permission.RECEIVE_SMS', 'com.android.launcher.permission.READ_SETTINGS', 'android.permission.WRITE_SETTINGS', 'com.google.android.providers.gsf.permission.READ_GSERVICES', 'android.permission.DOWNLOAD_WITHOUT_NOTIFICATION', 'android.permission.GET_TASKS', 'android.permission.WRITE_EXTERNAL_STORAGE', 'android.permission.RECORD_AUDIO', 'com.huawei.android.launcher.permission.CHANGE_BADGE', 'com.oppo.launcher.permission.READ_SETTINGS', 'android.permission.CHANGE_NETWORK_STATE', 'com.android.launcher.permission.INSTALL_SHORTCUT', 'android.permission.android.permission.READ_PHONE_STATE', 'android.permission.CALL_PHONE', 'android.permission.WRITE_CONTACTS', 'android.permission.READ_PHONE_STATE', 'com.samsung.android.providers.context.permi

In [26]:
# files.download("malware_rf_final.pkl")

# Flask

In [27]:
!mkdir flask_app
!mkdir flask_app/model


In [28]:
!mv malware_rf_final.pkl flask_app/model/


In [29]:
%%writefile flask_app/app.py

from flask import Flask, request, jsonify
import numpy as np
import pickle

app = Flask(__name__)

# ===== Load model =====
with open("/content/flask_app/model/malware_rf_final.pkl", "rb") as f:
    data = pickle.load(f)

model         = data["model"]
threshold     = data["threshold"]
feature_names = data["features"]

# ===== Predict endpoint =====
@app.route("/predict", methods=["POST"])
def predict():
    try:
        if not request.is_json:
            return error_response("INVALID_FORMAT", "Request must be JSON")

        content = request.get_json()

        if "features" not in content:
            return error_response("MISSING_FIELD", "'features' field is required")

        feature_dict = content["features"]

        missing = [f for f in feature_names if f not in feature_dict]
        if missing:
            return error_response("MISSING_FEATURE", f"Missing feature(s): {missing}")

        X = np.array(
            [feature_dict[f] for f in feature_names],
            dtype=float
        ).reshape(1, -1)

        prob = model.predict_proba(X)[0][1]
        pred = int(prob >= threshold)

        return jsonify({
            "status": "ok",
            "prediction": pred,
            "probability": round(float(prob), 4),
            "threshold": threshold
        })

    except ValueError:
        return error_response("INVALID_VALUE", "All features must be numeric")
    except Exception as e:
        return error_response("INTERNAL_ERROR", str(e))

# ===== Health check =====
@app.route("/health", methods=["GET"])
def health():
    return jsonify({"status": "ok"})

# ===== Error helper =====
def error_response(code, message, status=400):
    return jsonify({
        "status": "error",
        "error_code": code,
        "message": message
    }), status

if __name__ == "__main__":
    app.run(port=5000)

Writing flask_app/app.py


In [30]:
# !pkill -f flask
# !pkill -f cloudflared
# import time
# time.sleep(2)
# print("Stopped")

In [31]:
# !nohup python flask_app/app.py > flask.log 2>&1 &
# import time
# time.sleep(3)
# !curl http://localhost:5000/health

# تشغيل Flask: (كل جلسة)

!pkill -f flask
!pkill -f cloudflared
import time
time.sleep(2)
!nohup python flask_app/app.py > flask.log 2>&1 &
time.sleep(3)
!curl http://localhost:5000/health

{"status":"ok"}


In [32]:
# تشغيل cloudflared والحصول على الرابط: (كل جلسة)
!pkill -f cloudflared
import time
time.sleep(2)
!nohup ./cloudflared tunnel --url http://localhost:5000 > tunnel.log 2>&1 &
time.sleep(8)
!grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' tunnel.log | tail -1


# !nohup ./cloudflared tunnel --url http://localhost:5000 > tunnel.log 2>&1 &
# import time
# time.sleep(8)
# !grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' tunnel.log | tail -1

In [33]:
# import os
# print(os.path.exists("/content/flask_app/model/malware_rf_final.pkl"))
# print(os.path.exists("/content/flask_app/app.py"))

In [34]:
# import requests

# r = requests.get("https://potatoes-jacket-precisely-cabinet.trycloudflare.com/health")
# print(r.status_code, r.text)

In [36]:
# import requests

# # اختبار بيانات وهمية بنفس الـ feature names الحقيقية
# import pickle
# with open("/content/flask_app/model/malware_rf_final.pkl", "rb") as f:
#     data = pickle.load(f)

# feature_names = data["features"]
# print("Number of features:", len(feature_names))
# print("First 5:", feature_names[:5])

# # اختبار predict
# features = {f: 0 for f in feature_names}
# r = requests.post(
#     "https://potatoes-jacket-precisely-cabinet.trycloudflare.com/predict",
#     json={"features": features}
# )
# print(r.status_code, r.text)

In [37]:
# !cat flask.log

# NumPy


##  help us to learn what is the correct order of the conftion matrixes
  [[TN  FP]

  [FN  TP]]

In [ ]:
def extract_confusion_elements(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    TP = np.sum((y_true == 1) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    TN = np.sum((y_true == 0) & (y_pred == 0))

    return TP, FN, FP, TN

TP, FN, FP, TN = extract_confusion_elements(y_test, y_pred_tuned)

print("TP:", TP)
print("FN:", FN)
print("FP:", FP)
print("TN:", TN)

recall = TP / (TP + FN)
precision = TP / (TP + FP)

print("Recall:", recall)
print("Precision:", precision)

